# Data Science Job Market Analysis
- **Dataset:** ds_salaries.csv - 607 rows, Kaggle (2020-2023)
- **Tool:** DuckDB (in-process analytical SQL engine)
- **Goal:** Explore salary trends, remote work patterns, and experience-level dynamics in the data science job market.
  
## Table of Contents

  1. [Dataset Overview](#1-dataset-overview)  ||Check columns repartition ... Company location, Split of S/M/L, Split of FT & PT to see if some data shouldnt be use cuase insgnificant)||
  2. [Salary by Experience Level](#2-salary-by-experience-level)
  3. [Top Paying Job Titles](#3-top-paying-job-titles)
  4. [Salary Growth Year-over-Year](#4-salary-growth-year-over-year)
  5. [Remote Work Patterns](#5-remote-work-patterns)
  6. [Geographic Distribution](#6-geographic-distribution)
  7. [Company Size Impact](#7-company-size-impact)
  8. [Salary Rankings with Window Functions](#8-salary-rankings-with-window-functions)
  9. [Multi-Dimensional Analysis with CTEs](#9-multi-dimensional-analysis-with-ctes)

## 1. Dataset Overview

In [1]:
import duckdb
import pandas as pd

# Path to the dataset (relative to notebooks/ folder)
CSV = "../data/ds_salaries.csv"

# Quick sanity check to confirm the file loads correctly
result = duckdb.sql(f"SELECT COUNT(*) AS total_rows FROM '{CSV}'").df()
print(f"Dataset loaded: {result['total_rows'][0]} rows")

Dataset loaded: 607 rows


In [2]:
# Preview the first 5 rows and inspect column names
df_preview = duckdb.sql(f"SELECT * FROM '{CSV}' LIMIT 5").df()
df_preview

,column00,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


In [3]:
# Check for NULL values accross columns
null_preview = duckdb.sql(f"""
    SELECT
        (COUNT(*) - COUNT(work_year)) AS null_work_year,
        (COUNT(*) - COUNT(job_title)) AS null_job_title,
        (COUNT(*) - COUNT(experience_level)) AS null_experience_level,
        (COUNT(*) - COUNT(employment_type)) AS null_employment_type,
        (COUNT(*) - COUNT(salary)) AS null_salary,
        (COUNT(*) - COUNT(salary_currency)) AS null_salary_ccy,
        (COUNT(*) - COUNT(salary_in_usd)) AS null_salary_in_usd,
        (COUNT(*) - COUNT(employee_residence)) AS null_employee_residence,
        (COUNT(*) - COUNT(remote_ratio)) AS null_remote_ratio,
        (COUNT(*) - COUNT(company_location)) AS null_company_location,
        (COUNT(*) - COUNT(company_size)) AS null_company_size 
    FROM '{CSV}'
""").df()
null_preview.T

,0
null_work_year,0
null_job_title,0
null_experience_level,0
null_employment_type,0
null_salary,0
null_salary_ccy,0
null_salary_in_usd,0
null_employee_residence,0
null_remote_ratio,0
null_company_location,0


**Data Quality check:** No missing values detected across all 11 columns (607 rows).
The dataset is complete and requires no imputation or cleaning prior to analysis.

### Dataset Distribution

In [4]:
distribution = duckdb.sql(f"""
      SELECT 'experience_level' AS dimension, experience_level AS value, COUNT(*) AS count
      FROM '{CSV}' GROUP BY experience_level
      UNION ALL
      SELECT 'company_size', company_size, COUNT(*)
      FROM '{CSV}' GROUP BY company_size
      UNION ALL
      SELECT 'remote_ratio', CAST(remote_ratio AS VARCHAR), COUNT(*)
      FROM '{CSV}' GROUP BY remote_ratio
      UNION ALL
      SELECT 'company_location', company_location, COUNT(*)
      FROM '{CSV}' GROUP BY company_location HAVING COUNT(*) > 15
      ORDER BY dimension, count DESC
  """).df()
distribution

,dimension,value,count
0,company_location,US,355
1,company_location,GB,47
2,company_location,CA,30
3,company_location,DE,28
4,company_location,IN,24
5,company_size,M,326
6,company_size,L,198
7,company_size,S,83
8,experience_level,SE,280
9,experience_level,MI,213


**Data Distribution Outputs:**
- US dominates at 355/607 rows (58%)
- Executive level is sparse (26 rows)
- Remote ratio = 50 (hybrid) has only 99 rows vs 381 fully remote

## 2. Salary by Experience Level

In [5]:
salary_by_exp_level = duckdb.sql(f"""
    SELECT
        *,
        ROUND(avg_salary - LEAD(avg_salary) OVER(ORDER BY avg_salary DESC), 0) AS salary_jump_usd,
        ROUND(salary_jump_usd * 100.0 / avg_salary, 1) AS salary_jump_pct
    FROM (
        SELECT
            experience_level,
            COUNT(*) AS count_job,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 0) AS pct_jobs,
            ROUND(AVG(salary_in_usd), 0) AS avg_salary,
            ROUND(MIN(salary_in_usd), 0) AS min_salary,
            ROUND(MAX(salary_in_usd), 0) AS max_salary
        FROM '{CSV}'
        GROUP BY experience_level
    )
    ORDER BY avg_salary DESC
""").df()
salary_by_exp_level

,experience_level,count_job,pct_jobs,avg_salary,min_salary,max_salary,salary_jump_usd,salary_jump_pct
0,EX,26,4.0,199392.0,69741,600000,60775.0,30.5
1,SE,280,46.0,138617.0,18907,412000,50621.0,36.5
2,MI,213,35.0,87996.0,2859,450000,26353.0,29.9
3,EN,88,14.0,61643.0,4000,250000,NaN,NaN


**Key Findings:**
- The largest salary jump occurs between Mid-Level and Senior (36.5%).
- Entry-level roles represent only 14% of the dataset (88 out of 607 jobs), suggesting the market skews toward experienced professionals such as Senior & Mid Level.

## 3. Top Paying Job Titles

In [6]:
top_paying_job = duckdb.sql(f"""
    SELECT
        job_title,
        count(*) AS total_job,
        ROUND(AVG(salary_in_usd), 0) AS avg_salary_usd
    FROM '{CSV}'
    GROUP BY job_title
    HAVING total_job >= 5
    ORDER BY avg_salary_usd DESC
    LIMIT 10
    """).df()
top_paying_job

,job_title,total_job,avg_salary_usd
0,Principal Data Scientist,7,215242.0
1,Director of Data Science,7,195074.0
2,Data Architect,11,177874.0
3,Applied Data Scientist,5,175655.0
4,Head of Data,5,160163.0
5,Machine Learning Scientist,8,158413.0
6,Data Science Manager,12,158329.0
7,Lead Data Engineer,6,139725.0
8,Data Analytics Manager,7,127134.0
9,Data Engineering Manager,5,123227.0


**Key insights**:
- Top paying roles from 123K USD to 215K USD average salary (titles with fewer than 5 data points excluded).
- Specialist and principal-level positions command the highest salaries, such as Principal Data Scientist (215K USD), Director of Data Science (195K USD), Data Architect (178K USD).
- Management roles follow closely: Data Science Manager (158K USD), Lead Data Engineer (140K USD), Data Analytics Manager (127K USD).

## 4. Salary Growth Year-over-Year

In [7]:
salary_growth = duckdb.sql(f"""
    SELECT
        *,
        ROUND(avg_salary_usd - LAG(avg_salary_usd, 1) OVER(ORDER BY work_year ASC), 0) AS yoy_salary_jump,
        ROUND(yoy_salary_jump * 100.0 / avg_salary_usd) AS pct_yoy_salary_jump --duckdb allow re-use of alias
    FROM (
        SELECT
            work_year,
            COUNT(*) AS total_job,
            ROUND(AVG(salary_in_usd), 0) AS avg_salary_usd
        FROM '{CSV}'
        GROUP BY work_year
        )
    ORDER BY work_year ASC
    """).df()
salary_growth

,work_year,total_job,avg_salary_usd,yoy_salary_jump,pct_yoy_salary_jump
0,2020,72,95813.0,NaN,NaN
1,2021,217,99854.0,4041.0,4.0
2,2022,318,124522.0,24668.0,20.0


**Key Insights**
- Data science salaries grew modestly from 2020 to 2021 (+4%), then accelerated sharply in 2022 (+20%), reflecting the surge in market demand for data professionals post-pandemic.
- Dataset covers 2020-2022 only; 2023 trends are not represented.

## 5. Remote Work Patterns

In [8]:
remote_patterns = duckdb.sql(f"""
    SELECT
        CASE
            WHEN remote_ratio = 0 THEN 'On-Site'
            WHEN remote_ratio = 50 THEN 'Hybrid'                
            ELSE 'Fully Remote'
        END AS remote_type,
        COUNT(*) AS total_jobs,
        ROUND(COUNT(*) * 100.0 / sum(COUNT(*)) OVER(), 0) AS pct_jobs,
        ROUND(AVG(salary_in_usd), 0) AS avg_salary_usd
    FROM '{CSV}'
    GROUP BY remote_type
    ORDER BY avg_salary_usd DESC
    """).df()
remote_patterns

,remote_type,total_jobs,pct_jobs,avg_salary_usd
0,Fully Remote,381,63.0,122457.0
1,On-Site,127,21.0,106355.0
2,Hybrid,99,16.0,80823.0


**Key Insights**:
- Fully Remote roles dominate the dataset at 63%, followed by on-site (21%) and hybrid (16%).
- Remote roles also command highest average salary (122K USD), ahead of on-site (106K USD) and Hybrid (81K USD).

## 6. Geographic Distribution

In [9]:
geographic_distribution = duckdb.sql(f"""
    SELECT
        company_location,
        COUNT(*) as total_job,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 0) AS pct_jobs,
        ROUND(AVG(salary_in_usd), 0) AS avg_salary_usd
    FROM '{CSV}'
    GROUP BY company_location
    HAVING total_job >= 5
    ORDER BY avg_salary_usd DESC
    """).df()
geographic_distribution

,company_location,total_job,pct_jobs,avg_salary_usd
0,US,355,67.0,144055.0
1,JP,6,1.0,114127.0
2,CA,30,6.0,99824.0
3,DE,28,5.0,81887.0
4,GB,47,9.0,81583.0
5,FR,15,3.0,63971.0
6,ES,14,3.0,53060.0
7,GR,11,2.0,52293.0
8,IN,24,5.0,28582.0


**Key Insights:**
- United States dominates the dataset with 67% of the market for all jobs, followed by Great Britain (9%), Canada (6%) and Germany / India holding (5% each).
- The US also leads on compensation, averaging 144K USD - 26% above Japan (114K USD), the second highest paying market.
- European countries range from 52K USD (Greece) to 81K USD (Germany & Great Britain), while India ranks last at 28K USD average, reflecting significant regional salary disparities.
  

## 7. Company Size Impact

In [10]:
company_size_impact = duckdb.sql(f"""
    SELECT
        CASE
            WHEN company_size = 'S' THEN 'Small'
            WHEN company_size = 'M' THEN 'Medium'
            ELSE 'Large'
        END AS company_size,
        COUNT(*) AS total_job,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 0) as pct_job,
        ROUND(AVG(salary_in_usd), 0) AS avg_salary_usd
        FROM '{CSV}'
        GROUP BY company_size
        ORDER BY avg_salary_usd DESC
        """).df()
company_size_impact

,company_size,total_job,pct_job,avg_salary_usd
0,Large,198,33.0,119243.0
1,Medium,326,54.0,116905.0
2,Small,83,14.0,77633.0


**Key insights:**
- Medium companies dominate the market with 54% of the job positions and 116K USD average salary.
- Large companies hold 33% of the market with a slightly higher average salary of 119K USD compared to the Medium companies.
- Lastly, Small companies are 14% of the market and average salary ~33% down.

## 8. Salary Rankings with Window Functions

In [11]:
salary_rankings = duckdb.sql(f"""
    SELECT *
    FROM (
        SELECT
            CASE
                WHEN experience_level = 'EN' THEN '0 - Entry'
                WHEN experience_level = 'MI' THEN '1 - Middle'
                WHEN experience_level = 'SE' THEN '2 - Senior'
                ELSE '3 - Executive'
            END AS experience_level,
            job_title,
            COUNT(*) AS total_jobs,
            ROUND(AVG(salary_in_usd), 0) AS avg_salary_usd,
            RANK() OVER(PARTITION BY experience_level ORDER BY avg_salary_usd DESC) AS ranking
        FROM '{CSV}'
        GROUP BY experience_level,job_title
        HAVING COUNT(*) >=5
        )
    WHERE ranking <= 3
    ORDER BY experience_level,ranking
    """).df()
salary_rankings

,experience_level,job_title,total_jobs,avg_salary_usd,ranking
0,0 - Entry,Machine Learning Engineer,9,86996.0,1
1,0 - Entry,Data Science Consultant,5,62641.0,2
2,0 - Entry,Data Engineer,12,58934.0,3
3,1 - Middle,Research Scientist,7,136498.0,1
4,1 - Middle,Data Engineer,53,85986.0,2
5,1 - Middle,Data Scientist,60,82039.0,3
6,2 - Senior,Principal Data Scientist,5,187939.0,1
7,2 - Senior,Data Architect,8,182077.0,2
8,2 - Senior,Data Scientist,61,152971.0,3
9,3 - Executive,Director of Data Science,6,199586.0,1


**Key insights:**
- The top-ranked title at Senior level is Principal Data Scientist (188K USD), representing a 40% salary premium over its
  Mid-level equivalent (82K USD for Data Scientist).
- Executive roles are sparsely represented (6 jobs for Director of Data Science), but command the highest average
  salary in the dataset at 200K USD.
- Entry-level Machine Learning Engineers average USD 87K — above Mid-level Data Engineers (USD 86K) and Data Scientists
   (USD 82K), though this is based on only 9 observations and should be interpreted with caution.

## 9. Multi-Dimensional Analysis with CTEs

In [12]:
multi_dim_analysis = duckdb.sql(f"""
    WITH avg_by_segment AS (
        SELECT
            experience_level,
            company_size,
            COUNT(*) AS total_jobs,
            ROUND(AVG(salary_in_usd), 0) AS avg_salary_usd
        FROM '{CSV}'
        GROUP BY experience_level, company_size
    ),
    global_avg AS (
        SELECT
            ROUND(AVG(salary_in_usd), 0) AS avg_total_usd
        FROM '{CSV}'
        )
    SELECT 
        *,
        ROUND((avg_salary_usd - avg_total_usd) * 100.0 / avg_total_usd, 1) AS pct_vs_global_avg
    FROM avg_by_segment, global_avg
    ORDER BY experience_level, company_size
""").df()

multi_dim_analysis     

,experience_level,company_size,total_jobs,avg_salary_usd,avg_total_usd,pct_vs_global_avg
0,EN,L,29,72813.0,112298.0,-35.2
1,EN,M,30,50322.0,112298.0,-55.2
2,EN,S,29,62185.0,112298.0,-44.6
3,EX,L,11,221942.0,112298.0,97.6
4,EX,M,12,178242.0,112298.0,58.7
5,EX,S,3,201309.0,112298.0,79.3
6,MI,L,86,98030.0,112298.0,-12.7
7,MI,M,98,90091.0,112298.0,-19.8
8,MI,S,29,51159.0,112298.0,-54.4
9,SE,L,72,147591.0,112298.0,31.4


**Key Findings:**
  - Executive roles at large companies command nearly double the market average (+97.6%), making them the
  highest-compensated segment in the dataset.
  - Interestingly, Executive roles at small companies show a higher average (USD 201K vs USD 178K for medium), though this is based on only 3
  data points and should not be generalized.
  - Entry and Mid-level roles consistently fall below the global average, with Entry-level at medium companies reaching
  the lowest point at -55.2%.
  - Only Senior and Executive roles across all company sizes exceed the overall market average of USD 112K

**CONCLUSION:**
This analysis of 607 data science job records (2020–2022) highlights several key dynamics in the global job market:

  - **Experience drives salary more than any other factor.** The jump from Mid-level to Senior alone represents a 36.5%
  salary increase.
  - **The US dominates the market**, accounting for 67% of all positions and offering the highest average compensation at
   USD 144K.
  - **Remote work is the new standard**, with 63% of roles fully remote — and remote positions paying on average 15% more
   than on-site equivalents.
  - **Specialization pays at every level.** Entry-level ML Engineers outperform Mid-level Data Engineers and Data
  Scientists, confirming that technical depth commands a premium early in a career.
  - **Executive roles at large companies offer the highest compensation**, reaching nearly double the market average
  (+97.6%), while small companies surprisingly outpay medium firms for senior leadership.

  The dataset is limited to 3 years and skewed toward US-based companies, which should be considered when generalizing
  these findings to other geographies or more recent market conditions.
  Salary figures are sourced from a self-reported Kaggle dataset and may not reflect
  current market conditions.